In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=10000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [5]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescentLS(
    model=model,
    lr=1,
    c1=1e-4,
    tau=0.1,
    line_search_method="backtrack",
    line_search_cond="armijo"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.2525397837162018
epoch:  1, loss: 0.1701240837574005
epoch:  2, loss: 0.1198016107082367
epoch:  3, loss: 0.08875731378793716
epoch:  4, loss: 0.06948695331811905
epoch:  5, loss: 0.057479407638311386
epoch:  6, loss: 0.04997880756855011
epoch:  7, loss: 0.045285824686288834
epoch:  8, loss: 0.04234617203474045
epoch:  9, loss: 0.040503330528736115
epoch:  10, loss: 0.03934739902615547
epoch:  11, loss: 0.03862200677394867
epoch:  12, loss: 0.03816663473844528
epoch:  13, loss: 0.03788069263100624
epoch:  14, loss: 0.0377010852098465
epoch:  15, loss: 0.03758823126554489
epoch:  16, loss: 0.03751729428768158
epoch:  17, loss: 0.0374726727604866
epoch:  18, loss: 0.0374445840716362
epoch:  19, loss: 0.03742687776684761
epoch:  20, loss: 0.03741569072008133
epoch:  21, loss: 0.03740859776735306
epoch:  22, loss: 0.03740408271551132
epoch:  23, loss: 0.03740352764725685
epoch:  24, loss: 0.0374031700193882
epoch:  25, loss: 0.03740303963422775
epoch:  26, loss: 0.037398

In [6]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.7924763560295105
Test metrics:  R2 = 0.8109744787216187


In [7]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescentLS(
    model=model,
    lr=1,
    c1=1e-4,
    c2=0.9,
    tau=0.1,
    line_search_method="backtrack",
    line_search_cond="wolfe"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.36435964703559875
epoch:  1, loss: 0.34425878524780273
epoch:  2, loss: 0.32539892196655273
epoch:  3, loss: 0.3076964020729065
epoch:  4, loss: 0.29107439517974854
epoch:  5, loss: 0.2754627466201782
epoch:  6, loss: 0.26079681515693665
epoch:  7, loss: 0.24701647460460663
epoch:  8, loss: 0.234066441655159
epoch:  9, loss: 0.2218955159187317
epoch:  10, loss: 0.2104567289352417
epoch:  11, loss: 0.19970554113388062
epoch:  12, loss: 0.18960055708885193
epoch:  13, loss: 0.18010318279266357
epoch:  14, loss: 0.171177476644516
epoch:  15, loss: 0.16278961300849915
epoch:  16, loss: 0.15490806102752686
epoch:  17, loss: 0.1475035399198532
epoch:  18, loss: 0.14054825901985168
epoch:  19, loss: 0.13401705026626587
epoch:  20, loss: 0.12788499891757965
epoch:  21, loss: 0.12212823331356049
epoch:  22, loss: 0.11672437191009521
epoch:  23, loss: 0.11165303736925125
epoch:  24, loss: 0.10689426958560944
epoch:  25, loss: 0.10242960602045059
epoch:  26, loss: 0.09824190288

In [8]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.7183109521865845
Test metrics:  R2 = 0.7433587312698364


In [9]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescentLS(
    model=model,
    lr=10,
    c1=1e-4,
    c2=0.9,
    tau=0.1,
    line_search_method="backtrack",
    line_search_cond="strong-wolfe"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.16277511417865753
epoch:  1, loss: 0.10214575380086899
epoch:  2, loss: 0.07098507136106491
epoch:  3, loss: 0.05459709092974663
epoch:  4, loss: 0.04591701924800873
epoch:  5, loss: 0.04139700159430504
epoch:  6, loss: 0.03902817517518997
epoch:  7, loss: 0.03773145750164986
epoch:  8, loss: 0.0370035246014595
epoch:  9, loss: 0.036587558686733246
epoch:  10, loss: 0.036345258355140686
epoch:  11, loss: 0.03619951754808426
epoch:  12, loss: 0.03614933416247368
epoch:  13, loss: 0.0359409898519516
epoch:  14, loss: 0.035815298557281494
epoch:  15, loss: 0.03572379797697067
epoch:  16, loss: 0.0355130173265934
epoch:  17, loss: 0.03537847101688385
epoch:  18, loss: 0.03511445224285126
epoch:  19, loss: 0.03485497087240219
epoch:  20, loss: 0.034755196422338486
epoch:  21, loss: 0.03433570638298988
epoch:  22, loss: 0.0341019332408905
epoch:  23, loss: 0.0339617058634758
epoch:  24, loss: 0.033744122833013535
epoch:  25, loss: 0.03347313776612282
epoch:  26, loss: 0.03

In [10]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.553383469581604
Test metrics:  R2 = 0.6009252071380615


In [11]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescentLS(
    model=model,
    lr_init=1,
    c1=1e-4,
    c2=0.9,
    tau=0.1,
    line_search_method="backtrack",
    line_search_cond="goldstein"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.191791832447052
epoch:  1, loss: 0.191791832447052
epoch:  2, loss: 0.191791832447052
epoch:  3, loss: 0.191791832447052
epoch:  4, loss: 0.191791832447052
epoch:  5, loss: 0.191791832447052
epoch:  6, loss: 0.191791832447052
epoch:  7, loss: 0.191791832447052
epoch:  8, loss: 0.191791832447052
epoch:  9, loss: 0.191791832447052
epoch:  10, loss: 0.191791832447052
epoch:  11, loss: 0.191791832447052
epoch:  12, loss: 0.191791832447052
epoch:  13, loss: 0.191791832447052
epoch:  14, loss: 0.191791832447052
epoch:  15, loss: 0.191791832447052
epoch:  16, loss: 0.191791832447052
epoch:  17, loss: 0.191791832447052
epoch:  18, loss: 0.191791832447052
epoch:  19, loss: 0.191791832447052
epoch:  20, loss: 0.191791832447052
epoch:  21, loss: 0.191791832447052
epoch:  22, loss: 0.191791832447052
epoch:  23, loss: 0.191791832447052
epoch:  24, loss: 0.191791832447052
epoch:  25, loss: 0.191791832447052
epoch:  26, loss: 0.191791832447052
epoch:  27, loss: 0.191791832447052
ep

In [12]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = -6660.19970703125
Test metrics:  R2 = -7172.39501953125
